In [1]:
import kagglehub
import os
import tensorflow as tf
import keras
from keras import layers
from keras import ops
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Download latest version
root = kagglehub.dataset_download("muratkokludataset/pistachio-image-dataset")
print("Path to dataset files:", root)

100%|██████████| 26.7M/26.7M [00:00<00:00, 72.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/muratkokludataset/pistachio-image-dataset/versions/1


In [3]:
pistachio_image_dataset = os.path.join(root, "Pistachio_Image_Dataset")
os.listdir(pistachio_image_dataset)

['Pistachio_Image_Dataset_Request.txt',
 'Pistachio_16_Features_Dataset',
 'Pistachio_28_Features_Dataset',
 'Pistachio_Image_Dataset']

In [4]:
request_txt = os.path.join(pistachio_image_dataset, "Pistachio_Image_Dataset_Request.txt")
# with open(request_txt, "r") as f:
#     print(f.read())

In [5]:
image_dataset = os.path.join(pistachio_image_dataset, "Pistachio_Image_Dataset")
# os.listdir(image_dataset)

In [19]:
# @title Configurar os hiperparâmetros
learning_rate = 0.001
weight_decay = 0.0001
batch_size = 256
num_epochs = 30
BATCH_SIZE = 32
SEED = 42
IMG_SIZE = 180  # We'll resize input images to this size
patch_size = 6  # Size of the patches to be extract from the input images
num_patches = (IMG_SIZE // IMG_SIZE) ** 2
projection_dim = 64
num_heads = 4
transformer_units = [
    projection_dim * 2,
    projection_dim,
]  # Size of the transformer layers
transformer_layers = 8
mlp_head_units = [
    2048,
    1024,
]

In [20]:
train_ds = keras.utils.image_dataset_from_directory(image_dataset,
                                                            shuffle=True,
                                                            validation_split=0.2,
                                                            subset="training",
                                                            seed=SEED,
                                                            batch_size=BATCH_SIZE,
                                                            image_size=(IMG_SIZE, IMG_SIZE))

val_ds = keras.utils.image_dataset_from_directory(image_dataset,
                                                      shuffle=True,
                                                      validation_split=0.2,
                                                      subset="validation",
                                                      seed=SEED,
                                                      batch_size=BATCH_SIZE,
                                                      image_size=(IMG_SIZE, IMG_SIZE))

Found 2148 files belonging to 2 classes.
Using 1719 files for training.
Found 2148 files belonging to 2 classes.
Using 429 files for validation.


In [21]:
class_names = train_ds.class_names
num_classes = len(class_names)
print(class_names)

['Kirmizi_Pistachio', 'Siirt_Pistachio']


In [ ]:
plt.figure(figsize=(15, 12))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

In [23]:
# @title Data Augmentation

data_augmentation = keras.Sequential(
    [
        layers.Normalization(),
        layers.Resizing(IMG_SIZE, IMG_SIZE),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(factor=0.02),
        layers.RandomZoom(height_factor=0.2, width_factor=0.2),
    ],
    name="data_augmentation",
)
# Compute the mean and the variance of the training data for normalization.
# Iterate over the dataset to calculate the mean and variance
for images, _ in train_ds.take(1):
    data_augmentation.layers[0].adapt(images)

In [24]:
# @title Implement multilayer perceptron (MLP)

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=keras.activations.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [25]:
# @title Implement patching as a layer
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        input_shape = keras.ops.shape(images)
        batch_size = input_shape[0]
        height = input_shape[1]
        width = input_shape[2]
        channels = input_shape[3]
        num_patches_h = height // self.patch_size
        num_patches_w = width // self.patch_size
        patches = keras.ops.image.extract_patches(images, size=self.patch_size)
        patches = keras.ops.reshape(
            patches,
            (
                batch_size,
                num_patches_h * num_patches_w,
                self.patch_size * self.patch_size * channels,
            ),
        )
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config

In [ ]:
for image_batch, _ in train_ds.take(1):
    image = image_batch[0]
    break

plt.figure(figsize=(10, 10))
ax = plt.subplot(1,2)
plt.imshow(image.numpy().astype("uint8"))
plt.axis("off")

resized_image = ops.image.resize(
    ops.convert_to_tensor([image]), size=(IMG_SIZE, IMG_SIZE)
)
patches = Patches(patch_size)(resized_image)
print(f"Image size: {IMG_SIZE} X {IMG_SIZE}")
print(f"Patch size: {patch_size} X {patch_size}")
print(f"Patches per image: {patches.shape[1]}")
print(f"Elements per patch: {patches.shape[-1]}")

n = int(np.sqrt(patches.shape[1]))
plt.figure(figsize=(4, 4))
for i, patch in enumerate(patches[0]):
    ax = plt.subplot(n, n, i + 1)
    patch_img = ops.reshape(patch, (patch_size, patch_size, 3))
    plt.imshow(ops.convert_to_numpy(patch_img).astype("uint8"))
    plt.axis("off")

In [27]:
# @title Implement the patch encoding layer
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        batch_size = ops.shape(patch)[0]
        # Use tf.range instead of ops.range
        positions = ops.expand_dims(tf.range(start=0, limit=self.num_patches, delta=1), axis=0)
        positions = ops.repeat(positions, repeats=batch_size, axis=0) # Repeat for each batch
        projected_patches = self.projection(patch)
        encoded = projected_patches + self.position_embedding(positions)

        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({"num_patches": self.num_patches})
        return config

In [28]:
# @title Building the ViT model

def create_vit_classifier():
    input_shape = (IMG_SIZE, IMG_SIZE, 3)
    # Augment data.
    inputs = keras.Input(shape=input_shape)
    # Augment data.
    augmented = data_augmentation(inputs)
    # Create patches.
    patches = Patches(patch_size)(augmented)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.5)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    # Classify outputs.
    logits = layers.Dense(num_classes)(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)
    return model

In [29]:
def run_experiment(model):
    optimizer = keras.optimizers.AdamW(
        learning_rate=learning_rate, weight_decay=weight_decay
    )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            keras.metrics.SparseTopKCategoricalAccuracy(5, name="top-5-accuracy"),
        ],
    )

    checkpoint_filepath = "/tmp/checkpoint.weights.h5"
    checkpoint_callback = keras.callbacks.ModelCheckpoint(
        checkpoint_filepath,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    )

    history = model.fit(
        train_ds,
        batch_size=batch_size,
        epochs=num_epochs,
        validation_data=val_ds,
        callbacks=[checkpoint_callback],
    )

    model.load_weights(checkpoint_filepath)
    _, accuracy, top_5_accuracy = model.evaluate(val_ds)
    print(f"Test accuracy: {round(accuracy * 100, 2)}%")
    print(f"Test top 5 accuracy: {round(top_5_accuracy * 100, 2)}%")

    return history

In [30]:
vit_classifier = create_vit_classifier()
history = run_experiment(vit_classifier)

Epoch 1/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.5409 - loss: 26.1183 - top-5-accuracy: 1.0000 - val_accuracy: 0.5501 - val_loss: 2.3988 - val_top-5-accuracy: 1.0000
Epoch 2/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 98s 2s/step - accuracy: 0.5934 - loss: 3.8888 - top-5-accuracy: 1.0000 - val_accuracy: 0.6131 - val_loss: 0.5979 - val_top-5-accuracy: 1.0000
Epoch 3/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 50s 923ms/step - accuracy: 0.6274 - loss: 1.1849 - top-5-accuracy: 1.0000 - val_accuracy: 0.5688 - val_loss: 0.7206 - val_top-5-accuracy: 1.0000
Epoch 4/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.5962 - loss: 1.0083 - top-5-accuracy: 1.0000 - val_accuracy: 0.8228 - val_loss: 0.5459 - val_top-5-accuracy: 1.0000
Epoch 5/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 103s 959ms/step - accuracy: 0.6021 - loss: 0.7412 - top-5-accuracy: 1.0000 - val_accuracy: 0.6597 - val_loss: 0.5813 - val_top-5-accuracy: 1.0000
Epoch 6/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 80s 921ms/step - accuracy: 0.6398 - loss: 0.6571 - to

In [ ]:
def plot_history(item):
    plt.plot(history.history[item], label=item)
    plt.plot(history.history["val_" + item], label="val_" + item)
    plt.xlabel("Epochs")
    plt.ylabel(item)
    plt.title("Train and Validation {} Over Epochs".format(item), fontsize=14)
    plt.legend()
    plt.grid()
    plt.show()


plot_history("loss")
plot_history("top-5-accuracy")